# Part 4: Text Generation

Your model is trained. Now let's make it write.

Text generation with a GPT is **autoregressive**: generate one token at a time, append it to the input, and repeat.

This notebook walks through the generation algorithm and all its knobs — greedy decoding, temperature, and top-k sampling.

## Setup

In [ ]:
!pip install -q torch

import torch
import torch.nn as nn
from dataclasses import dataclass

print(f"PyTorch: {torch.__version__}")

## Load the Model

Copy in the model architecture (same as Parts 2 & 3), then load a trained checkpoint.

> **Don't have a checkpoint?** Run the training notebook first (Part 3), or use a pretrained checkpoint. The cell after the model definition shows how to download one.

In [ ]:
@dataclass
class GPTConfig:
    vocab_size: int = 65
    block_size: int = 256
    n_layer: int = 6
    n_head: int = 6
    n_embd: int = 384

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.n_head = config.n_head
        self.n_embd = config.n_embd

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        head_dim = C // self.n_head
        q = q.view(B, T, self.n_head, head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, head_dim).transpose(1, 2)
        y = torch.nn.functional.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu = nn.GELU(approximate='tanh')
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)

    def forward(self, x):
        return self.c_proj(self.gelu(self.c_fc(x)))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(0, T, device=idx.device)
        x = self.transformer.wte(idx) + self.transformer.wpe(pos)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = nn.functional.cross_entropy(
                logits.view(-1, logits.size(-1)), targets.view(-1)
            )
        return logits, loss

print("Model classes defined.")

In [ ]:
# Load a trained checkpoint
# Update this path to point to your checkpoint file
CHECKPOINT_PATH = "checkpoint_final.pt"

checkpoint = torch.load(CHECKPOINT_PATH, weights_only=False, map_location="cpu")
config = checkpoint["config"]
stoi = checkpoint["stoi"]
itos = checkpoint["itos"]

model = GPT(config)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print(f"Loaded checkpoint from step {checkpoint['step']}")
print(f"Config: {config.n_layer}L/{config.n_head}H/{config.n_embd}D, vocab={config.vocab_size}")

## The Naive Approach: Greedy Decoding

Always pick the most probable next token. This is deterministic — the same prompt always produces the same output. It tends to be repetitive because the highest-probability continuation reinforces itself.

In [ ]:
@torch.no_grad()
def generate_greedy(model, prompt, stoi, itos, max_new_tokens=200):
    device = next(model.parameters()).device
    tokens = [stoi[c] for c in prompt if c in stoi]
    idx = torch.tensor([tokens], dtype=torch.long, device=device)

    model.eval()
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -model.config.block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :]          # last position only
        next_token = logits.argmax(dim=-1, keepdim=True)
        idx = torch.cat([idx, next_token], dim=1)

    return "".join([itos[i] for i in idx[0].tolist()])

print("Greedy output:")
print(generate_greedy(model, "To be or not", stoi, itos, max_new_tokens=150))

## Temperature

Scale the logits before applying softmax. The math: softmax computes `exp(logit_i) / sum(exp(logit_j))`. Dividing all logits by temperature changes the distribution:

- **T → 0**: approaches greedy (argmax) — very deterministic
- **T = 1.0**: normal probabilities
- **T > 1.0**: flattens the distribution, giving rare tokens more chance
- **T = 0.7–0.9**: the typical sweet spot for coherent but varied text

In [ ]:
import matplotlib.pyplot as plt
import torch.nn.functional as F

# Visualize how temperature reshapes the probability distribution
torch.manual_seed(0)
logits = torch.randn(10)  # simulated logits for 10 tokens

temps = [0.3, 0.7, 1.0, 1.5, 2.0]
fig, axes = plt.subplots(1, len(temps), figsize=(14, 3), sharey=True)

for ax, t in zip(axes, temps):
    probs = F.softmax(logits / t, dim=-1).numpy()
    ax.bar(range(len(probs)), probs)
    ax.set_title(f"T = {t}")
    ax.set_xlabel("Token ID")

axes[0].set_ylabel("Probability")
plt.suptitle("Effect of Temperature on Token Distribution", y=1.02)
plt.tight_layout()
plt.show()

## Top-k Sampling

Only consider the `k` most probable tokens. Set everything else to `-inf` so softmax gives them zero probability.

This prevents the model from sampling extremely unlikely tokens. With a character-level model (vocab=65), `top_k=40` is reasonable — it still considers most characters but excludes the very unlikely ones.

In [ ]:
# Visualize top-k filtering
torch.manual_seed(0)
logits = torch.randn(20)  # 20 token vocab

fig, axes = plt.subplots(1, 3, figsize=(14, 3))

for ax, k in zip(axes, [20, 10, 5]):
    filtered = logits.clone()
    if k < len(logits):
        values, _ = torch.topk(filtered, k)
        filtered[filtered < values[-1]] = float("-inf")
    probs = F.softmax(filtered, dim=-1).numpy()
    bars = ax.bar(range(len(probs)), probs)
    # Color zeroed-out tokens
    for i, p in enumerate(probs):
        if p == 0:
            bars[i].set_color('lightgray')
    ax.set_title(f"top_k = {k}")
    ax.set_xlabel("Token ID")

axes[0].set_ylabel("Probability")
plt.suptitle("Effect of top-k Filtering (gray = excluded)", y=1.02)
plt.tight_layout()
plt.show()

## The Full Generate Function

The pipeline for each token:
1. Run the model on the current sequence → get logits for next position
2. Apply temperature scaling
3. Filter with top-k (remove very unlikely tokens)
4. Convert to probabilities with softmax
5. Sample from the distribution with `multinomial`
6. Append the sampled token and repeat

`@torch.no_grad()` disables gradient computation — we don't need it for inference and it saves memory.

In [ ]:
@torch.no_grad()
def generate(model, prompt, stoi, itos, max_new_tokens=200, temperature=0.8, top_k=40):
    device = next(model.parameters()).device
    tokens = [stoi[c] for c in prompt if c in stoi]
    idx = torch.tensor([tokens], dtype=torch.long, device=device)

    model.eval()
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -model.config.block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature

        if top_k > 0:
            values, _ = torch.topk(logits, top_k)
            logits[logits < values[:, -1:]] = float("-inf")

        probs = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_token], dim=1)

    return "".join([itos[i] for i in idx[0].tolist()])

## Experiment: Temperature & Top-k

Try different settings with the same prompt and see how output changes.

In [ ]:
PROMPT = "To be or not to be"

settings = [
    ("Very deterministic", 0.1, 40),
    ("Balanced (default)", 0.8, 40),
    ("Creative",           1.2, 40),
    ("Narrow top-k=5",     0.8,  5),
]

for label, temp, k in settings:
    torch.manual_seed(42)  # same seed for fair comparison
    output = generate(model, PROMPT, stoi, itos,
                      max_new_tokens=120, temperature=temp, top_k=k)
    print(f"\n=== {label} (temperature={temp}, top_k={k}) ===")
    print(output)

## Reproducibility with Seeds

Generation involves random sampling (`torch.multinomial`), so the same prompt produces different output each time. Set a seed for reproducible results.

In [ ]:
# Same seed → same output every time
for run in range(3):
    torch.manual_seed(42)
    output = generate(model, "To be or not", stoi, itos,
                      max_new_tokens=80, temperature=0.8)
    print(f"Run {run+1}: {output[:80]}...")

print("\n--- Different seeds ---")
for seed in [1, 2, 3]:
    torch.manual_seed(seed)
    output = generate(model, "To be or not", stoi, itos,
                      max_new_tokens=80, temperature=0.8)
    print(f"Seed {seed}: {output[:80]}...")

## Cherry-Picking: Generate Many, Keep the Best

Generate multiple samples and pick your favorite. This is completely fair game — GPT-4 uses similar sampling to give users better outputs.

In [ ]:
prompt = "Shall I compare"
n_samples = 5

print(f"Generating {n_samples} samples from prompt: '{prompt}'\n")

for i in range(n_samples):
    output = generate(model, prompt, stoi, itos,
                      max_new_tokens=150, temperature=0.85, top_k=40)
    print(f"--- Sample {i+1} ---")
    print(output)
    print()

## Key Takeaways

- Autoregressive generation: predict one token, append, repeat
- Greedy decoding is deterministic and repetitive
- Temperature controls randomness — `0.7–0.9` is usually the sweet spot
- Top-k removes extremely unlikely tokens from consideration
- Set a seed to reproduce a specific output
- Generate many samples and cherry-pick — it's standard practice

**Next: [Part 5 — Putting It All Together](05-putting-it-together.ipynb)**